# Week 11 — Class 2: Outlier Detection and Treatment
**Charlotte Project | Interactive teaching notebook | ~180 minutes**

**Outcome:** identify, explain and document unusual measurements using Z-scores, IQR and domain validation. **A statistical flag is not an automatic deletion instruction.**

Today we use a **synthetic retail dataset** to learn. The separately assessed lab uses a **public-derived wine chemistry dataset with deliberately injected errors**. Do not complete the lab in this teaching notebook.

**Learning pathway:** concept → worked example → run and predict → interpret → decision and audit trail.

## Section A — What is an outlier? (30 minutes)

An **outlier** is a value that looks unusually distant from the main pattern. It can arise from data entry mistakes, measurement failures, rare but legitimate transactions, or a change in the underlying process. A high-value transaction might be a fraud signal, a wholesale order, or a typing error. None of these can be established by a formula alone.

**Three distinct questions:** (1) Is the value numerically unusual? (2) Is the value impossible under known domain rules? (3) What action is appropriate given the intended analysis?

**Why it matters:** automatic deletion can bias a model or remove the most important records. Always preserve original values, trace flags to row IDs, and record treatment decisions.

**Before running:** predict whether `510` and `0` are necessarily errors. What further metadata would you request?

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

path = Path('retail_outliers_SYNTHETIC.csv')
if not path.exists():
    path = Path('teaching/retail_outliers_SYNTHETIC.csv')
retail = pd.read_csv(path)
print(retail[['order_id','order_value','quantity']].to_string(index=False))
print('Median:', retail['order_value'].median(), 'Mean:', round(retail['order_value'].mean(), 2))

order_id  order_value  quantity
     R01           49         2
     R02           52         3
     R03           50         2
     R04           53         3
     R05           48         2
     R06           51         3
     R07           49         2
     R08           52         2
     R09           50         3
     R10           51         2
     R11          510         2
     R12            0         0
Median: 50.5 Mean: 84.58


**Interpretation:** `510` pulls the mean upward. `0` might represent an uncompleted order, a free sample, or an error. The source process determines the correct response. A summary statistic does not prove invalidity.

**Try:** change `510` to `110` in a temporary copy and compare mean and median. Do not overwrite the original dataset.

In [2]:
trial = retail.copy()
trial.loc[trial['order_value']==510,'order_value']=110
print('Original mean:',round(retail['order_value'].mean(),2),'New mean:',round(trial['order_value'].mean(),2))

Original mean: 84.58 New mean: 51.25


## Section B — Z-scores (35 minutes)

A **Z-score** measures how many standard deviations a value is from the mean: `z=(value−mean)/standard_deviation`.

In a roughly symmetric distribution without severe contamination, unusually large absolute Z-scores can provide useful flags. For this course use population standard deviation (`ddof=0`) and `abs(z)>3` for a *screening* rule.

**Limitations:** extreme values distort both mean and standard deviation, which can hide themselves. For small samples, skewed distributions or groups with zero variance, this simple rule may perform poorly. A value with `z=2.9` is not automatically valid.

**Worked example:** if mean=50 and SD=10, then 80 has z=3.0. Under strict `>3` it is *not* flagged; clarify whether a threshold is strict or inclusive.

In [3]:
series = retail['order_value'].astype(float)
mean, sd = series.mean(), series.std(ddof=0)
retail['z_score'] = (series - mean) / sd if sd > 0 else 0.0
retail['z_flag'] = retail['z_score'].abs() > 3
print('Mean:',round(mean,2),'SD:',round(sd,2))
print(retail[['order_id','order_value','z_score','z_flag']].to_string(index=False))

Mean: 84.58 SD: 129.03
order_id  order_value   z_score  z_flag
     R01           49 -0.275784   False
     R02           52 -0.252533   False
     R03           50 -0.268034   False
     R04           53 -0.244782   False
     R05           48 -0.283534   False
     R06           51 -0.260283   False
     R07           49 -0.275784   False
     R08           52 -0.252533   False
     R09           50 -0.268034   False
     R10           51 -0.260283   False
     R11          510  3.297136    True
     R12            0 -0.655552   False


**Common mistakes:** using `std()` without specifying `ddof`, comparing signed Z-scores rather than `abs(z)`, and assuming a Z-score is a probability. `std(ddof=0)` is a consistent convention for the exercise; neither convention is universally right for all statistical inference.

**Try:** create a group where every value equals 5. Why must we handle zero variance?

In [4]:
constant = pd.Series([5.,5.,5.])
sd0=constant.std(ddof=0)
flags = pd.Series(False,index=constant.index) if sd0==0 else ((constant-constant.mean())/sd0).abs()>3
print('Constant SD:',sd0,'Flags:',flags.tolist())

Constant SD: 0.0 Flags: [False, False, False]


## Section C — Interquartile Range (IQR) (45 minutes)

Quartiles divide sorted values into four parts. **Q1** is the 25th percentile; **Q3** is the 75th. The IQR is `Q3−Q1`, describing the middle half of the observations.

A common screening rule marks values outside `[Q1−1.5×IQR, Q3+1.5×IQR]`. This is less influenced by extreme values than using mean and SD, though it may flag legitimate observations in skewed distributions.

**Exact quartiles depend on calculation conventions.** Our code uses Pandas `Series.quantile`, by default linear interpolation. Never combine thresholds computed with different conventions without documenting them.

**Prediction:** Do the Z-score and IQR methods have to flag the same values? No. Run the example and compare.

In [5]:
q1,q3=series.quantile(.25),series.quantile(.75)
iqr=q3-q1
low,high=q1-1.5*iqr,q3+1.5*iqr
retail['iqr_flag']=(series<low)|(series>high)
print('Q1:',q1,'Q3:',q3,'IQR:',iqr,'Fences:',low,high)
print(retail[['order_id','order_value','z_flag','iqr_flag']].to_string(index=False))

Q1: 49.0 Q3: 52.0 IQR: 3.0 Fences: 44.5 56.5
order_id  order_value  z_flag  iqr_flag
     R01           49   False     False
     R02           52   False     False
     R03           50   False     False
     R04           53   False     False
     R05           48   False     False
     R06           51   False     False
     R07           49   False     False
     R08           52   False     False
     R09           50   False     False
     R10           51   False     False
     R11          510    True      True
     R12            0   False      True


**Discussion:** Which of the rules labels more values? Is that because one method is inherently correct? Consider sample size, skew, and whether order values differ by customer type.

**Try:** split by sales channel and calculate channel-specific quartiles. With very small groups, warn that the fences may become unstable.

In [6]:
def fences(s):
    q1,q3=s.quantile([.25,.75]); v=q3-q1
    return pd.Series({'lower':q1-1.5*v,'upper':q3+1.5*v,'n':len(s)})
print(retail.groupby('channel')['order_value'].apply(fences).unstack().round(2))

         lower  upper    n
channel                   
online    45.5   55.5  6.0
store     45.5   55.5  6.0


## Section D — Treatment choices and audit trails (35 minutes)

Possible responses include **retain with a flag**, **investigate and correct against a trusted source**, **quarantine clear invalid records**, **use robust statistics**, or **cap values with explicit analytical justification**. Unusual does not equal invalid.

**Decision sequence:** (1) preserve raw value and ID; (2) compute screening flags; (3) evaluate documented validity rules; (4) separate confirmed invalids from unverified unusual observations; (5) write traceable outputs; (6) review with someone who understands the domain.

For the retail exercise, we'll assume *only* negative order values are invalid. No negative values appear, so no records are deleted; statistical anomalies remain visible for review. This is an **illustrative policy**, not a general business rule.

In [7]:
retail['domain_invalid']=retail['order_value'] < 0
retail['review_needed']=retail[['z_flag','iqr_flag','domain_invalid']].any(axis=1)
retained=retail.loc[~retail['domain_invalid']].copy()
quarantined=retail.loc[retail['domain_invalid']].copy()
print('Rows retained:',len(retained),'Quarantined:',len(quarantined))
print('Review IDs:',retail.loc[retail['review_needed'],'order_id'].tolist())

Rows retained: 12 Quarantined: 0
Review IDs: ['R11', 'R12']


**ML connection (preview):** when preprocessing for machine learning, calculate thresholds on the training fold, then apply these frozen parameters to other folds. Calculating them over all data beforehand risks leakage. Detailed pipelines come later.

**General data literacy:** the same validation principles apply to spreadsheets, JSON event logs, SQL tables and Parquet datasets, though their typing and ingestion issues differ.

In [8]:
print('Quality report')
print(pd.DataFrame({'metric':['input_rows','z_flags','iqr_flags','domain_invalid','review_needed'],
                    'value':[len(retail),int(retail['z_flag'].sum()),int(retail['iqr_flag'].sum()),
                             int(retail['domain_invalid'].sum()),int(retail['review_needed'].sum())]}).to_string(index=False))

Quality report
        metric  value
    input_rows     12
       z_flags      1
     iqr_flags      2
domain_invalid      0
 review_needed      2


## Section E — Knowledge checks, AWS connection and GitHub (35 minutes)

**Knowledge checks:**
1. Why might a genuine unusual transaction be more important than an average one?
2. Why may Z-score fail on extremely skewed small samples?
3. What does IQR measure, and which quantile convention does our code use?
4. When may a record be quarantined under an explicit domain validity rule?
5. Why should raw observations remain unchanged in the audit trail?

**AWS DEA-C01 connections — mini-scenarios:**
- A recurring pipeline produces suspicious measurements. Where should consistent data-quality rules be enforced and recorded? Discuss AWS Glue data-quality concepts and monitoring.
- A record violates a documented ingestion rule. Should the pipeline silently drop it, or preserve evidence for review?
- If records are statistically extreme but plausible, should an ETL transformation delete them automatically?

These are **supporting certification concepts**; AWS service deployment is not a requirement for this class.

**GitHub checkpoint:** implement the separate lab as scripts and tests in your individual repository; branch `feature/week11-class02-outliers`, commit, push and open a PR. Notebook examples are practice, not the assessed submission.

**Exit ticket:** explain one reason to *retain* a flagged value and one reason to *quarantine* an invalid one.